# SeqTrainer Shared-Split Model Baselines

Run CNN reference, CNN-v2, DNABERT2 frozen embeddings, iPro-MP FASTA preparation, and benchmark comparison on the same train/validation/test promoter CSVs.

Scientific rules used here:
- the same split files are used for every model
- thresholds are selected on validation only
- test metrics are final reporting only
- primary comparison metrics are MCC and AUPRC
- missing heavy model dependencies are allowed to skip with a manifest rather than fake metrics

## 1. Setup repository

In [ ]:
from pathlib import Path
import os
import sys

REPO_URL = "https://github.com/simplyshree/SeqTrainer.git"
BRANCH = "issue-3-all-model-baselines"
REPO_DIR = Path("/content/SeqTrainer")

if REPO_DIR.exists():
    %cd /content/SeqTrainer
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull --ff-only origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
    %cd /content/SeqTrainer

print("Repository:", REPO_DIR)
!git rev-parse --abbrev-ref HEAD
!git rev-parse --short HEAD

## 2. Install package

DNABERT2 needs `torch`, `transformers`, and the Hugging Face model files. On Colab T4, installing from the repo extra is the clean baseline.

In [ ]:
!python -m pip install --upgrade pip setuptools wheel -q
!python -m pip install -e ".[torch]" -q

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import seqtrainer
print("SeqTrainer import ok")

## 3. Mount Drive and copy shared dataset

This uses the same CNN/DNABERT2/iPro-MP split files from Drive:

`/content/drive/MyDrive/AIxBio/Promoter Classification/Data`

In [ ]:
from pathlib import Path
import shutil

DRIVE_DATA_DIR = Path("/content/drive/MyDrive/AIxBio/Promoter Classification/Data")
LOCAL_DATA_DIR = REPO_DIR / "data" / "promoter_classification"
LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
    else:
        print("Drive already mounted")
except Exception as exc:
    print("Drive mount skipped or failed:", exc)

split_files = {
    "train": "train_EP_DNA_BERT2_genomic_order.csv",
    "validation": "eval_EP_DNA_BERT2_genomic_order.csv",
    "test": "test_EP_DNA_BERT2_genomic_order.csv",
}

missing = []
for split, file_name in split_files.items():
    source = DRIVE_DATA_DIR / file_name
    target = LOCAL_DATA_DIR / file_name
    if source.exists():
        shutil.copy2(source, target)
        print(f"Copied {split}: {source} -> {target}")
    elif target.exists():
        print(f"Already local {split}: {target}")
    else:
        missing.append((split, source, target))

if missing:
    print("Missing files:")
    for split, source, target in missing:
        print(f"- {split}: expected Drive {source} or local {target}")
    raise FileNotFoundError("Could not find all required shared split CSVs.")

!ls -lh data/promoter_classification

## 4. Inspect split sizes and labels

In [ ]:
import pandas as pd

for split, file_name in split_files.items():
    frame = pd.read_csv(LOCAL_DATA_DIR / file_name)
    print("\n", split, file_name)
    print("rows:", len(frame))
    print("columns:", list(frame.columns))
    print("label counts:")
    print(frame["label"].value_counts(dropna=False).sort_index())
    print("sequence length summary:")
    print(frame["sequence"].astype(str).str.len().describe())

## 5. Run CNN reference baseline

In [ ]:
from seqtrainer.benchmarks.runner import run_benchmark

CNN_CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "cnn.toml"
cnn_result = run_benchmark(CNN_CONFIG, base_dir=REPO_DIR, allow_skip=False)
print("status:", cnn_result.status)
print("output_dir:", cnn_result.output_dir)

## 6. Run CNN-v2 improved baseline

In [ ]:
CNN_V2_CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "cnn_v2.toml"
cnn_v2_result = run_benchmark(CNN_V2_CONFIG, base_dir=REPO_DIR, allow_skip=False)
print("status:", cnn_v2_result.status)
print("output_dir:", cnn_v2_result.output_dir)

## 7. Enable DNABERT2 download for Colab

The repo config defaults to local-only model loading for reproducibility. In Colab, this cell writes a local runtime config copy with `allow_download = true` so Hugging Face can fetch DNABERT2.

In [ ]:
DNABERT2_CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "dnabert2_frozen.toml"
DNABERT2_COLAB_CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "dnabert2_frozen_colab.toml"

text = DNABERT2_CONFIG.read_text(encoding="utf-8")
text = text.replace("allow_download = false", "allow_download = true")
DNABERT2_COLAB_CONFIG.write_text(text, encoding="utf-8")
print(DNABERT2_COLAB_CONFIG)

import inspect
import importlib
import seqtrainer.torch.dnabert2_benchmark as dnabert2_benchmark
importlib.reload(dnabert2_benchmark)
source_text = inspect.getsource(dnabert2_benchmark)
print("DNABERT2 runner file:", dnabert2_benchmark.__file__)
print("Has state-dict fallback:", "_load_dnabert2_from_state_dict" in source_text)
print("Has meta check:", "_meta_parameter_names" in source_text)

## 8. Run DNABERT2 frozen embedding baseline

This is the first DNABERT2 model to compare against CNN-v2. It freezes the encoder, caches embeddings, trains a small classifier head, and uses validation MCC for threshold selection.

In [ ]:
dnabert2_result = run_benchmark(DNABERT2_COLAB_CONFIG, base_dir=REPO_DIR, allow_skip=False)
print("status:", dnabert2_result.status)
print("output_dir:", dnabert2_result.output_dir)

## 9. Prepare iPro-MP FASTA files

This does not train iPro-MP. It writes FASTA files so iPro-MP/iPromoter can be run externally on the same split rows.

In [ ]:
IPROMP_CONFIG = REPO_DIR / "config-examples" / "benchmarks" / "ipromp_external.toml"
ipromp_result = run_benchmark(IPROMP_CONFIG, base_dir=REPO_DIR)
print("status:", ipromp_result.status)
print("output_dir:", ipromp_result.output_dir)
print("skip_reason:", ipromp_result.manifest.get("extra", {}).get("skip_reason"))

## 10. Compare completed runs

The comparison ranks held-out test MCC first and test AUPRC second. Skipped iPro-MP manifests do not contribute metrics until external predictions are added.

In [ ]:
from seqtrainer.benchmarks import compare_benchmark_outputs

artifact_dirs = [cnn_result.output_dir, cnn_v2_result.output_dir, dnabert2_result.output_dir]
comparison = compare_benchmark_outputs(
    artifact_dirs,
    output_dir=REPO_DIR / "outputs" / "benchmarks" / "comparison",
)
print(comparison)
pd.read_csv(comparison["comparison_metrics"]).sort_values(["split", "mcc", "auprc"], ascending=[True, False, False]).head(20)

## 11. Download artifacts

In [ ]:
!zip -qr /content/seqtrainer_model_baseline_outputs.zip outputs/benchmarks
print("Created /content/seqtrainer_model_baseline_outputs.zip")